In [1]:
from pyspark.sql import SparkSession

from utility import data_preparation,preprocessing_normal

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# Create SparkSession 
spark = SparkSession.builder.master("local[8]").appName("all-doc-pairs-similarity.com").config("spark.driver.memory", "10g").getOrCreate()

sc = spark.sparkContext

23/05/22 16:47:09 WARN Utils: Your hostname, michele-Veriton-M2631G resolves to a loopback address: 127.0.1.1; using 192.168.1.6 instead (on interface enp3s0)
23/05/22 16:47:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/05/22 16:47:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
corpus, keys = data_preparation("nfcorpus")

/home/michele/Documents/All-Documents-Pairs-Similarity/datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:…

  0%|          | 0/3633 [00:00<?, ?it/s]

In [4]:
#vectorized_docs_rdd=preprocessing_spark()
vectorized_docs_rdd=preprocessing_normal(corpus,keys,sc)

documents cleaning:   0%|          | 0/3633 [00:00<?, ?it/s]

In [5]:
#vectorized_docs_rdd.collect()

In [6]:
def Map(doc_pair):
    
    id, doc = doc_pair
    
    non_zero_terms=doc.nonzero()
    
    term__doc_ids=[]
    
    for term_index in non_zero_terms[1]:
        if doc.argmax()==term_index:
            term__doc_ids.append((term_index,id))
    
    return term__doc_ids

term_listDocIds_pairs_rdd=vectorized_docs_rdd.flatMap(Map).groupByKey()

In [7]:
vectorized_docs_broadcast=sc.broadcast(dict(vectorized_docs_rdd.collect()))

In [8]:
import itertools

threshold=sc.broadcast(0.8)

def Reduce(id_list):
    
    similarities=[]
    
    for id1, id2 in itertools.combinations(id_list,2):

        similarity=vectorized_docs_broadcast.value[id1].dot(vectorized_docs_broadcast.value[id2].transpose())
        
        if similarity[0,0]>=threshold.value:
            similarities.append(((id1,id2),similarity[0,0]))
        
    return similarities

similar_doc_pairs=term_listDocIds_pairs_rdd.values().flatMap(Reduce).persist()

In [9]:
similar_doc_pairs.collect()

[(('MED-398', 'MED-3207'), 0.9999999999999999),
 (('MED-4988', 'MED-4990'), 1.0000000000000002),
 (('MED-2944', 'MED-4617'), 0.9999999999999996),
 (('MED-118', 'MED-2651'), 0.9999999999999997),
 (('MED-306', 'MED-2910'), 1.0000000000000004),
 (('MED-2769', 'MED-5359'), 0.9999999999999997),
 (('MED-2170', 'MED-4977'), 0.9999999999999994),
 (('MED-4255', 'MED-4613'), 1.0000000000000009),
 (('MED-1106', 'MED-4820'), 0.8421713117238266),
 (('MED-2189', 'MED-2205'), 0.9999999999999996),
 (('MED-2155', 'MED-5244'), 1.0),
 (('MED-4247', 'MED-4616'), 1.0000000000000004),
 (('MED-3485', 'MED-4393'), 1.0000000000000002),
 (('MED-4599', 'MED-4674'), 0.9999999999999996),
 (('MED-4633', 'MED-5342'), 1.0),
 (('MED-3885', 'MED-3886'), 0.9975824231771594),
 (('MED-2951', 'MED-4689'), 1.0),
 (('MED-3193', 'MED-3787'), 0.9999999999999999),
 (('MED-3220', 'MED-3235'), 0.9999999999999993),
 (('MED-3811', 'MED-4416'), 0.9999999999999996),
 (('MED-2977', 'MED-4892'), 1.0000000000000004),
 (('MED-2100', 'MED

In [10]:
dict(zip(keys,corpus))['2452989']

KeyError: '2452989'

In [ ]:
dict(zip(keys,corpus))['39506601']

'Novel roles for KLF1 in erythropoiesis revealed by mRNA-seq. KLF1 (formerly known as EKLF) regulates the development of erythroid cells from bi-potent progenitor cells via the transcriptional activation of a diverse set of genes. Mice lacking Klf1 die in utero prior to E15 from severe anemia due to the inadequate expression of genes controlling hemoglobin production, cell membrane and cytoskeletal integrity, and the cell cycle. We have recently described the full repertoire of KLF1 binding sites in vivo by performing KLF1 ChIP-seq in primary erythroid tissue (E14.5 fetal liver). Here we describe the KLF1-dependent erythroid transcriptome by comparing mRNA-seq from Klf1(+/+) and Klf1(-/-) erythroid tissue. This has revealed novel target genes not previously obtainable by traditional microarray technology, and provided novel insights into the function of KLF1 as a transcriptional activator. We define a cis-regulatory module bound by KLF1, GATA1, TAL1, and EP300 that coordinates a core s

In [ ]:
#spark.stop()